<a href="https://colab.research.google.com/github/hamzafarooq/multi-agent-course/blob/main/modules/Module_3_Production_Agentic_RAG_AI_Systems/003.%20Agentic%20Router_semantic_caching_rbac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic RAG with Semantic Cache and Access Control

This notebook combines three ideas:

- **Semantic cache** — stores past questions (as vectors) and their answers. If a new question *means* the same as a stored one, the stored answer is returned instantly, with no LLM or API call. This is the same cache you built in notebook 002, now packaged in `rag_helpers.py`.
- **Agentic RAG (routing)** — an LLM (**GPT-5.6-Luna**) reads each question and picks the right knowledge source: OpenAI documentation (Qdrant), 10-K financial filings (Qdrant), or live internet search (SerpApi).
- **Role-based access control (RBAC)** — checks whether a user's role is allowed to read a file *before* anything is retrieved. Denied requests are rejected immediately, with no embedding, search or LLM call.

## Architecture — two separate entry points

The notebook has **two functions you can call**, and they take different paths.

**Path A — `agentic_rag_with_cache(query, cache)`** (Section 5): cache + router, no access control.

```
User query
    │
    ▼
┌─────────────────────────────┐
│  Is query time-sensitive?   │──YES──▶ Router → source ──▶ answer
│  ("today", "now", "latest", │         (answer is NOT cached)
│   "stock price", ...)       │
└──────────────┬──────────────┘
               │ NO
               ▼
┌─────────────────────────────┐
│   Semantic cache lookup     │──HIT──▶ Return cached answer ⚡
│   (FAISS nearest neighbour) │
└──────────────┬──────────────┘
               │ MISS
               ▼
┌─────────────────────────────┐
│          Router             │
│  (GPT-5.6-Luna picks one)   │
└──────┬──────────┬───────────┘
       │          │           │
  OPENAI_QUERY 10K_DOC_QUERY INTERNET_QUERY
       │          │           │
    Qdrant      Qdrant      SerpApi
  (OpenAI docs) (10-Ks)    (live web)
       │          │           │
       └──────────┴───────────┘
               │
               ▼
   Store answer in cache 💾  (only if it succeeded — errors are never cached)
               │
               ▼
          Return answer
```

**Path B — `secure_agentic_rag(user, file, query)`** (Section 6): access control, no cache, no router.

```
User query + user_id + file_id
    │
    ▼
┌─────────────────────────────┐
│  RBAC check                 │──DENIED──▶ 🚫 Reject (no data touched)
│  user → role → allowed file?│
└──────────────┬──────────────┘
               │ ALLOWED
               ▼
  Retrieve from THAT file only
  (Qdrant collection, or the inline runbook text)
               │
               ▼
          Return answer
```

In Path B the caller already names the file, so there is nothing to route. It also skips the cache on purpose: the cache stores answers by question only, not by who asked. If Path B shared that cache, an answer fetched for a finance analyst could be served to an engineer who asks a similar question. The two paths are kept separate in this notebook; combining them safely is one of the extensions at the end.

## Why combine them?

- **Speed** — a cached answer returns in milliseconds instead of the several seconds a full RAG call takes.
- **Cost** — fewer LLM and API calls for repeated or similar questions.
- **Correctness** — time-sensitive questions (e.g. *"What is the current stock price of Apple?"*) never use the cache, so they always get a fresh answer.
- **Security** — in the RBAC path, the access check runs before any retrieval or LLM call, so unauthorised requests never reach the data or cost anything.

## 1. Setup

Install the libraries, clone the course repository (it contains `rag_helpers.py` and the pre-built Qdrant vector database), and import the helper functions.

In [1]:
# Dependencies. Safe to skip on a re-run if the environment is already set up —
# the setup cell below is separate on purpose, so skipping this one costs you nothing.
!pip install -q -U pip setuptools
!pip install -q "transformers==4.48.0" "sentence-transformers==3.4.1" "einops==0.8.1" faiss-cpu openai qdrant_client python-dotenv nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 14.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# Setup — always run this one.
import os, sys, shutil, nest_asyncio

# The Nomic model ships remote code; a stale copy in the HF cache causes confusing
# load errors, so clear it and let it re-download.
_nomic_cache = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules/nomic-ai")
if os.path.exists(_nomic_cache):
    shutil.rmtree(_nomic_cache)
    print("Cleared cached Nomic remote code — it will be re-downloaded fresh.")

try:
    import google.colab
    _REPO = "/content/multi-agent-course"
    if not os.path.exists(_REPO):
        os.system("git clone https://github.com/hamzafarooq/multi-agent-course.git")
        print("Repository cloned ✅")
    else:
        print("Repository already present ✅")
    _MODULE_DIR = f"{_REPO}/modules/Module_3_Production_Agentic_RAG_AI_Systems"
except ImportError:
    # Running locally — rag_helpers.py sits next to this notebook
    _MODULE_DIR = os.getcwd()
    print(f"Running locally — helpers path: {_MODULE_DIR}")

sys.path.insert(0, _MODULE_DIR)
nest_asyncio.apply()  # Required for asyncio.run() inside Jupyter/Colab

Repository cloned ✅


In [3]:
from rag_helpers import init_rag, SemanticCaching, agentic_rag_with_cache
print("✅ Helpers imported from rag_helpers.py")

✅ Helpers imported from rag_helpers.py


## 2. API Keys

**On Google Colab** — store keys in the Secrets panel (`🔑` icon, left sidebar):
| Secret name | Where to get it |
|---|---|
| `SERP_API_KEY` | [serpapi.com](https://serpapi.com) |
| `OPENAI_API_KEY` | [platform.openai.com](https://platform.openai.com) |

**Running locally** — add keys to `Module_3_Production_Agentic_RAG_AI_Systems/.env`:
```
serp_api_key=<your_key>
openai_api_key=<your_key>
```
The cell below detects the environment automatically and loads from the right source.

In [4]:
# ── Load API keys ─────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    serp_api_key   = userdata.get('SERP_API_KEY')
    openai_api_key = userdata.get('OPENAI_API_KEY')
    QDRANT_PATH    = f"{_REPO}/modules/Module_3_Production_Agentic_RAG_AI_Systems/Agentic_RAG/qdrant_data"
    print("Colab: credentials loaded from Secrets.")
except ImportError:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    serp_api_key   = os.getenv("serp_api_key") or os.getenv("SERP_API_KEY") or os.getenv("SERPAPI_KEY")
    openai_api_key = os.getenv("openai_api_key") or os.getenv("OPENAI_API_KEY")
    QDRANT_PATH    = os.path.join(_MODULE_DIR, "Agentic_RAG", "qdrant_data")
    print("Local: credentials loaded from .env.")

print(f"SerpApi key:    {'✅' if serp_api_key else '❌ MISSING'}")
print(f"OpenAI API key: {'✅' if openai_api_key else '❌ MISSING'}")

# ── Initialise the RAG pipeline (loads models + connects to Qdrant) ───────────
init_rag(openai_api_key=openai_api_key, serp_api_key=serp_api_key, qdrant_path=QDRANT_PATH)

Colab: credentials loaded from Secrets.
SerpApi key:    ✅
OpenAI API key: ✅
Loading Nomic text model for Qdrant retrieval embeddings...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- configuration_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_hf_nomic_bert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- modeling_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/547M [00:00<?, ?B/s]

✅ RAG pipeline ready.


## 3. Create the Semantic Cache

`SemanticCaching` is defined in `rag_helpers.py`. It only **looks up** and **stores** answers; the decision of what to do on a hit or miss is made by `agentic_rag_with_cache()` in Section 5.

| Method | What it does |
|---|---|
| `is_time_sensitive(q)` | Returns `True` if the question contains a time word such as "today", "now", "latest" or "stock price" (matched as whole words, so "know" does not count as "now"). These questions never use the cache. |
| `check_cache(q)` | Embeds the question, finds the closest stored question with FAISS, and returns `(hit, answer, embedding, similarity, row_id)`. The embedding is returned even on a miss so it doesn't have to be computed twice. |
| `add_to_cache(q, answer, embedding)` | Saves a new question + answer to the FAISS index and to the JSON file. |

**Threshold (`threshold=0.30`)** — FAISS reports the *squared* distance between the two question vectors. A distance ≤ 0.30 counts as a hit. Lower = stricter.
This is the value measured in notebook 002: real paraphrases landed at ~0.16–0.25, while related-but-different questions were ~0.38 and above. At 0.2 some genuine paraphrases were missed.

The "similarity" printed on a hit is the **cosine similarity** between the two questions (1.0 = identical).

In [5]:
# Create the semantic cache.
# threshold=0.30 is the value measured in notebook 002 (also the default in rag_helpers.py).
# clear_on_init=True wipes any entries left over from a previous run.
cache = SemanticCaching(json_file='rag_cache.json', threshold=0.30, clear_on_init=True)

Loading Nomic embedding model for semantic cache...


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Cache embedding model ready.
Semantic cache cleared.


## 4. Agentic RAG Pipeline (from `rag_helpers.py`)

All the pipeline pieces live in `rag_helpers.py`. You don't call most of them directly; `agentic_rag_with_cache()` uses them for you.

| Function | What it does |
|---|---|
| `route_query(query)` | Asks **GPT-5.6-Luna** to label the question `OPENAI_QUERY`, `10K_DOCUMENT_QUERY` or `INTERNET_QUERY`. Falls back to `INTERNET_QUERY` if anything goes wrong. |
| `get_internet_content(query)` | Live Google search through SerpApi (used for `INTERNET_QUERY`). |
| `_retrieve_and_respond(query, action)` | Embeds the question → gets the 3 closest chunks from the matching Qdrant collection → asks GPT-5.6-Luna to answer **only** from those chunks, citing them as [1], [2], [3]. |
| `_run_rag_pipeline(query)` | Runs the router, then the matching handler. Returns the answer plus a flag saying whether it succeeded. |
| **`agentic_rag_with_cache(query, cache)`** | **Main entry point** — adds the cache on top of the pipeline. |

**Qdrant collections loaded by `init_rag()`:**
- `opnai_data` — OpenAI Agents documentation
- `10k_data` — 10-K filings for **both** Uber (2021) and Lyft (2024), in one collection

**Errors are never cached.** If a search or retrieval fails (network error, no results, etc.), you see an error message, but nothing is stored. Otherwise one temporary failure would be served to every similar question afterwards.

**Why the Nomic model is loaded twice.** The cache embeds questions with `SentenceTransformer` (normalised vectors). The Qdrant collections were built with a different method (raw model output, averaged, not normalised), so the retriever embeds queries that same way to match. Each index gets queries embedded the way its own data was embedded.

## 5. Demo — Path A: Semantic Cache + Agentic RAG

`agentic_rag_with_cache(query, cache)` is the only function to call here. It does the time check, cache lookup, routing, retrieval, caching and printing.

### Test queries

| # | Query | Expected path |
|---|---|---|
| 1 | *"What was Uber's revenue in 2021?"* | Cache MISS → router → 10-K Qdrant → stored |
| 2 | *"How much did Uber earn in fiscal year 2021?"* | Cache HIT (same meaning as #1) |
| 3 | *"How do I build an agent with the OpenAI Agents SDK?"* | Cache MISS → router → OpenAI docs Qdrant → stored |
| 4 | *"What are the best AI tools this week?"* | Time-sensitive ("this week") → skip cache → router (likely internet) → **not** stored |
| 5 | *"What is the current stock price of Apple?"* | Time-sensitive ("current", "stock price") → skip cache → router (likely internet) → **not** stored |
| 6 | *"What are the most popular open-source LLMs?"* | Cache MISS → router → internet (SerpApi) → stored |
| 7 | *"Which open-source large language models are most widely used?"* | Cache HIT (same meaning as #6) |

Two things to keep in mind:
- **Time-sensitive questions still go through the router.** Being time-sensitive only means "don't use the cache"; it doesn't force an internet search. A question like *"What was Uber's revenue last year?"* is time-sensitive but would still be routed to the 10-K collection.
- **The route and the cache hits can vary.** The router is an LLM, and whether a paraphrase hits depends on the exact distance. Watch the printed route and the hit/miss line rather than assuming the table.

In [6]:
# Test 1: Cache MISS — router should pick 10K_DOCUMENT_QUERY; the answer is stored
result = agentic_rag_with_cache("What was Uber's revenue in 2021?", cache)

👤 Query: What was Uber's revenue in 2021?

❌ Cache MISS — running Agentic RAG pipeline...

📍 Route: 10K_DOCUMENT_QUERY  |  Asks about Uber's reported 2021 revenue.

💾 Cached for future similar queries.

🤖 Response:
Uber's revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]



In [7]:
# Test 2: Cache HIT — same meaning as Test 1, returned instantly from the cache
result = agentic_rag_with_cache("How much did Uber earn in fiscal year 2021?", cache)

👤 Query: How much did Uber earn in fiscal year 2021?

✅ Cache HIT (row 0, similarity: 0.838, 0.243s)

🤖 Response (cached):
Uber's revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]



In [8]:
# Test 3: Cache MISS — router should pick OPENAI_QUERY; the answer is stored
result = agentic_rag_with_cache("How do I build an agent with the OpenAI Agents SDK?", cache)

👤 Query: How do I build an agent with the OpenAI Agents SDK?

❌ Cache MISS — running Agentic RAG pipeline...

📍 Route: OPENAI_QUERY  |  Asks about building an agent using the OpenAI Agents SDK.

💾 Cached for future similar queries.

🤖 Response:
To build an agent with the OpenAI Agents SDK, define three core components:

1. **Model** — the language model responsible for reasoning and decisions  
2. **Tools** — functions or APIs the agent can call  
3. **Instructions** — behavioral guidelines and guardrails for the agent [1]

### 1. Install the SDK

```bash
pip install openai-agents
```

Set your API key:

```bash
export OPENAI_API_KEY="your-api-key"
```

### 2. Define a tool

Tools let the agent interact with external systems or perform actions.

```python
from agents import function_tool

@function_tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    # Replace this with a real weather API call.
    return f"The weather in {city} is sunny and 22°C

In [9]:
# Test 4: Time-sensitive ("this week") — skips the cache; the router decides the source
# (most likely INTERNET_QUERY → SerpApi). The answer is NOT stored.
result = agentic_rag_with_cache("What are the best AI tools this week?", cache)

👤 Query: What are the best AI tools this week?

⏰ Time-sensitive — bypassing cache for a fresh answer.

📍 Route: INTERNET_QUERY  |  Asks about current AI tool recommendations.
Getting your response from the internet 🌐 ...

🤖 Response (live):
[1] 19 Best AI Tools for Work in 2026
    Best AI tools for work that solve daily workflow problems. Canva Magic Studio Runway ML Adobe Firefly Designs.ai. Superhuman Lavender Grammarly ...
    Source: https://www.rokform.com/blogs/rokform-blog/best-ai-tools-for-work?srsltid=AU7gw4Vazl0AW0Mar9Rfd9jbEGKR4qzz2zZAn7q0552S6epQfN11lzf8

[2] There are hundreds of new AI tools released every week, ...
    OpenAI O3 for deep research—It. Sonnet 3.7 & Gemini 2.5 Pro for code generation—GitHub. AWS Bedrock for inference—
    Source: https://www.linkedin.com/posts/apoorv93singh_there-are-hundreds-of-new-ai-tools-released-activity-7328050924092907529-KJ2Q

[3] AI Tools Every Content Creator Needs in 2026 (The Stack ...
    1. ChatGPT — Your Creative Business P

In [10]:
# Test 5: Time-sensitive ("current", "stock price") — always answered fresh, never stored
result = agentic_rag_with_cache("What is the current stock price of Apple?", cache)

👤 Query: What is the current stock price of Apple?

⏰ Time-sensitive — bypassing cache for a fresh answer.

📍 Route: INTERNET_QUERY  |  Requests current stock market data for Apple.
Getting your response from the internet 🌐 ...

🤖 Response (live):
[1] Apple (AAPL) CFDs
    Track the latest Apple stock price at 330.65 with a -0.53% daily change (indicative prices). Buy/sell Apple (AAPL) share CFDs.
    Source: https://www.plus500.com/en-au/instruments/aapl

[2] Apple Inc. Stock (APC) - Quote Xetra - MarketScreener
    Fifteen Years at Apple's Helm: Taking Stock of Tim Cook's Tenure ... Current year, 208.45. Extreme 208.45. 301.5. 1 year, 208.15. Extreme 208.15. 301.5.
    Source: https://www.marketscreener.com/quote/stock/APPLE-INC-438411/

[3] Apple Share Prices & Stock Live Charts (AAPL.US)
    Commissions $0.00. Est. Share Price. $335.69. FXCM Micronization.
    Source: https://www.fxcm.com/za/quotes/shares/aapl/

[4] Apple Inc. (AAPL) Stock Price | Live Quotes & Charts
    Get lates

In [11]:
# Test 6: Cache MISS — router should pick INTERNET_QUERY (SerpApi); the answer is stored
result = agentic_rag_with_cache("What are the most popular open-source LLMs?", cache)

👤 Query: What are the most popular open-source LLMs?

❌ Cache MISS — running Agentic RAG pipeline...

📍 Route: INTERNET_QUERY  |  Asks about general popularity of open-source LLMs
Getting your response from the internet 🌐 ...

💾 Cached for future similar queries.

🤖 Response:
[1] Large language model
    They are the basis for many modern chatbots, such as ChatGPT, Claude, Gemini, Grok, and DeepSeek. LLMs are typically based on transformer architecture.
    Source: https://en.wikipedia.org/wiki/Large_language_model

[2] What Are Large Language Models (LLMs)?
    LLMs are easily accessible to the public through interfaces like Anthropic's Claude, Open AI's ChatGPT, Microsoft's Copilot, Meta's Llama models ...
    Source: https://www.ibm.com/think/topics/large-language-models

[3] Large Language Models (LLMs) with Google AI
    Google AI's Veo, Imagen and Chirp are examples of such models that will spawn new applications and help create solutions to the world's most challenging ...
    S

In [12]:
# Test 7: Cache HIT — same meaning as Test 6
result = agentic_rag_with_cache("Which open-source large language models are most widely used?", cache)

👤 Query: Which open-source large language models are most widely used?

❌ Cache MISS — running Agentic RAG pipeline...

📍 Route: INTERNET_QUERY  |  Asks about general adoption of open-source large language models.
Getting your response from the internet 🌐 ...

💾 Cached for future similar queries.

🤖 Response:
[1] Best Open Source LLMs in 2026 (Updated for Q3) - QbitNeural
    For many AI teams, closed-source models like the GPT-5.6 family (Sol, Terra, and Luna) and Claude Fable 5 are convenient.
    Source: https://qbitneural.com/open-source-llms/

[2] Top 10 Open Source and Paid LLMs
    Leading LLMs for Various Use Cases · GPT-3 — Versatile for a wide range of tasks. · GPT-4 — Enhanced version of GPT-3. · BERT — Effective for ...
    Source: https://medium.com/@salma.elhaimer028/top-10-open-source-and-paid-llms-a-comprehensive-guide-9f676c359e92

[3] What are your Top 3 Large Language Models (open ...
    Here are my top picks: GPT-2 by OpenAI: been open-sourced, BERT by Google: LLaM

## 6. Path B — Role-Based Access Control (RBAC)

So far, anyone could ask anything and reach any knowledge source. In a real company, different people should see different documents: an engineer probably shouldn't read finance's 10-K analysis, and a finance analyst doesn't need the internal on-call runbook.

This section adds a small **RBAC layer**. Every request says *who* is asking and *which file* they want. The user's role is checked first:
- **Denied** → rejected immediately. Nothing is embedded, searched or sent to an LLM.
- **Allowed** → the answer is retrieved from **that one file only**.

This path is separate from Section 5: it does **not** use the router (the caller already names the file) and does **not** use the semantic cache (the cache doesn't know who asked, so sharing it could leak one role's answers to another).

**Setup for this demo — 2 users, 2 roles, 3 files:**

| File | `engineer` (alice) | `finance_analyst` (bob) |
|---|---|---|
| 📘 OpenAI Agents Documentation (shared) | ✅ | ✅ |
| 📗 10-K Filings — Uber 2021 & Lyft 2024 (finance-only) | ❌ | ✅ |
| 🛠️ Internal On-Call Runbook (engineering-only) | ✅ | ❌ |

The OpenAI docs and the 10-K filings reuse the two Qdrant collections already loaded by `init_rag()`. Note that the 10-K "file" is the **whole** `10k_data` collection, so it covers both Uber and Lyft. (The notes at the end of this section show how to restrict it to one company.)

The on-call runbook is a short made-up document written directly in the code below. It shows a third, engineering-only source without building a new Qdrant collection.

In [13]:
# ── File registry ────────────────────────────────────────────────────────────
# Each file says where its content lives: one of the two existing Qdrant
# collections, or (for the runbook) a small inline mock document.
FILES = {
    "openai_docs": {
        "title": "OpenAI Agents Documentation",
        "source": "qdrant",
        "action": "OPENAI_QUERY",        # → Qdrant collection 'opnai_data'
    },
    "10k_filings": {
        "title": "10-K Filings (Uber 2021 & Lyft 2024)",
        "source": "qdrant",
        "action": "10K_DOCUMENT_QUERY",  # → Qdrant collection '10k_data' (both companies)
    },
    "internal_runbook": {
        "title": "Internal Engineering On-Call Runbook",
        "source": "mock",
        "content": """
            ON-CALL RUNBOOK — Payments Service
            1. Page the on-call engineer via PagerDuty if the error rate exceeds
               2% for more than 5 minutes.
            2. Open the #incidents Slack channel and start a war room if it's a P1.
            3. Roll back the most recent deploy with `deploy rollback payments-service`.
            4. A postmortem is required for any P1 or P2 incident within 48 hours.
        """,
    },
}

# ── Roles → the files each role is allowed to read ─────────────────────────
ROLE_PERMISSIONS = {
    "engineer":        {"openai_docs", "internal_runbook"},
    "finance_analyst": {"openai_docs", "10k_filings"},
}

# ── Users → role ─────────────────────────────────────────────────────────────
# Stand-in for a real identity provider (SSO/JWT claims, an internal users
# table, etc.) — swap this out for a real lookup in production.
USERS = {
    "alice": "engineer",
    "bob":   "finance_analyst",
}


def has_access(user_id: str, file_id: str) -> bool:
    """True only if the user's role is explicitly allowed to read this file.
    Unknown users and unknown roles are denied (allow-list, deny by default)."""
    role = USERS.get(user_id)
    return role is not None and file_id in ROLE_PERMISSIONS.get(role, set())

In [14]:
import asyncio
import rag_helpers  # shared OpenAI client, model name, and the Qdrant retriever


def _answer_from_mock_doc(user_query: str, doc_text: str) -> str:
    """Answer from a small local document (the runbook) — no Qdrant needed."""
    prompt = f"""
    Based ONLY on the following internal document, answer the user's question.
    If the document doesn't contain the answer, say so explicitly.

    Document:
    {doc_text}

    Question: {user_query}
    """
    response = rag_helpers._openaiclient.chat.completions.create(
        model=rag_helpers.LLM_MODEL,   # same model as the router and RAG answers
        messages=[{"role": "system", "content": prompt}],
    )
    return response.choices[0].message.content


def secure_agentic_rag(user_id: str, file_id: str, user_query: str) -> str:
    """
    Access-controlled retrieval (Path B).

    1. Check the user's role against the requested file.
    2. Denied → return a refusal. Nothing is embedded, searched or sent to an LLM.
    3. Allowed → answer from that one file only.

    This path does not use the router or the semantic cache.
    """
    CYAN, RED, GREEN, BOLD, RESET = "\033[96m", "\033[91m", "\033[92m", "\033[1m", "\033[0m"

    role      = USERS.get(user_id, "UNKNOWN")
    file_meta = FILES.get(file_id)

    print(f"{BOLD}{CYAN}👤 User:{RESET} {user_id}  (role: {role})")
    print(f"{BOLD}{CYAN}📄 Requested file:{RESET} {file_meta['title'] if file_meta else file_id}")

    if file_meta is None:
        print(f"{RED}❌ Unknown file: {file_id}{RESET}\n")
        return f"❌ Unknown file: {file_id}"

    if not has_access(user_id, file_id):
        print(f"{RED}🚫 ACCESS DENIED{RESET} — role '{role}' has no permission for this file.\n")
        return (
            f"🚫 Access denied: your role ('{role}') does not have permission "
            f"to view '{file_meta['title']}'."
        )

    print(f"{GREEN}✅ Access granted{RESET} — retrieving...\n")

    try:
        if file_meta["source"] == "mock":
            result = _answer_from_mock_doc(user_query, file_meta["content"])
        else:
            result = asyncio.run(rag_helpers._retrieve_and_respond(user_query, file_meta["action"]))
    except Exception as e:   # rag_helpers raises RAGError when retrieval fails
        result = f"⚠️ {e}"

    print(f"{BOLD}{CYAN}🤖 Response:{RESET}\n{result}\n")
    return result

### Demo — same questions, different roles

Each question below is sent by both `alice` (engineer) and `bob` (finance_analyst). The same request is allowed for a file the role may read and denied for a file it may not.

In [15]:
print("="*70)
print("1) alice (engineer) asks about the OpenAI docs — SHARED file → ALLOWED")
print("="*70)
secure_agentic_rag("alice", "openai_docs", "How do I build an agent with the OpenAI Agents SDK?")


1) alice (engineer) asks about the OpenAI docs — SHARED file → ALLOWED
👤 User: alice  (role: engineer)
📄 Requested file: OpenAI Agents Documentation
✅ Access granted — retrieving...

🤖 Response:
An agent built with the OpenAI Agents SDK has three core parts:

1. **Model** — the LLM that reasons and makes decisions  
2. **Tools** — functions or APIs the agent can call  
3. **Instructions** — rules that define the agent’s behavior [1]

### 1. Install the SDK

```bash
pip install openai-agents
```

Set your API key:

```bash
export OPENAI_API_KEY="your-api-key"
```

### 2. Define a tool

```python
from agents import function_tool

@function_tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    # Replace this with a real weather API call.
    return f"The weather in {city} is sunny and 22°C."
```

### 3. Create the agent

```python
from agents import Agent

weather_agent = Agent(
    name="Weather agent",
    instructions=(
        "You are a helpful 

'An agent built with the OpenAI Agents SDK has three core parts:\n\n1. **Model** — the LLM that reasons and makes decisions  \n2. **Tools** — functions or APIs the agent can call  \n3. **Instructions** — rules that define the agent’s behavior [1]\n\n### 1. Install the SDK\n\n```bash\npip install openai-agents\n```\n\nSet your API key:\n\n```bash\nexport OPENAI_API_KEY="your-api-key"\n```\n\n### 2. Define a tool\n\n```python\nfrom agents import function_tool\n\n@function_tool\ndef get_weather(city: str) -> str:\n    """Return the current weather for a city."""\n    # Replace this with a real weather API call.\n    return f"The weather in {city} is sunny and 22°C."\n```\n\n### 3. Create the agent\n\n```python\nfrom agents import Agent\n\nweather_agent = Agent(\n    name="Weather agent",\n    instructions=(\n        "You are a helpful assistant who talks to users about the weather. "\n        "Use the get_weather tool when current weather information is needed."\n    ),\n    tools=[get_we

In [16]:
print("="*70)
print("2) alice (engineer) asks about the on-call runbook — ENGINEER-ONLY → ALLOWED")
print("="*70)
secure_agentic_rag("alice", "internal_runbook", "What should I do if the error rate spikes?")


2) alice (engineer) asks about the on-call runbook — ENGINEER-ONLY → ALLOWED
👤 User: alice  (role: engineer)
📄 Requested file: Internal Engineering On-Call Runbook
✅ Access granted — retrieving...

🤖 Response:
⚠️ module 'rag_helpers' has no attribute 'LLM_MODEL'



"⚠️ module 'rag_helpers' has no attribute 'LLM_MODEL'"

In [17]:
print("="*70)
print("3) alice (engineer) asks about the 10-K filings — FINANCE-ONLY → DENIED")
print("="*70)
secure_agentic_rag("alice", "10k_filings", "What was Uber's revenue in 2021?")

3) alice (engineer) asks about the 10-K filings — FINANCE-ONLY → DENIED
👤 User: alice  (role: engineer)
📄 Requested file: 10-K Filings (Uber 2021 & Lyft 2024)
🚫 ACCESS DENIED — role 'engineer' has no permission for this file.



"🚫 Access denied: your role ('engineer') does not have permission to view '10-K Filings (Uber 2021 & Lyft 2024)'."

In [18]:
print("="*70)
print("4) bob (finance_analyst) asks about the 10-K filings — FINANCE-ONLY → ALLOWED")
print("="*70)
secure_agentic_rag("bob", "10k_filings", "What was Uber's revenue in 2021?")

4) bob (finance_analyst) asks about the 10-K filings — FINANCE-ONLY → ALLOWED
👤 User: bob  (role: finance_analyst)
📄 Requested file: 10-K Filings (Uber 2021 & Lyft 2024)
✅ Access granted — retrieving...

🤖 Response:
Uber's revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]



"Uber's revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]"

In [19]:
print("="*70)
print("5) bob (finance_analyst) asks about the OpenAI docs — SHARED file → ALLOWED")
print("="*70)
secure_agentic_rag("bob", "openai_docs", "How do I build an agent with the OpenAI Agents SDK?")


5) bob (finance_analyst) asks about the OpenAI docs — SHARED file → ALLOWED
👤 User: bob  (role: finance_analyst)
📄 Requested file: OpenAI Agents Documentation
✅ Access granted — retrieving...

🤖 Response:
An agent in the OpenAI Agents SDK is built from three core components:

1. **Model** — the language model that reasons and makes decisions  
2. **Tools** — functions or APIs the agent can call  
3. **Instructions** — rules describing the agent’s behavior [1]

### 1. Install the SDK

```bash
pip install openai-agents
```

Set your API key:

```bash
export OPENAI_API_KEY="your-api-key"
```

### 2. Define a tool

Tools let the agent take actions or retrieve information from external systems.

```python
from agents import function_tool

@function_tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    # Replace this with a real weather API call.
    return f"The weather in {city} is sunny and 22°C."
```

### 3. Create an agent

```python
from agents im

'An agent in the OpenAI Agents SDK is built from three core components:\n\n1. **Model** — the language model that reasons and makes decisions  \n2. **Tools** — functions or APIs the agent can call  \n3. **Instructions** — rules describing the agent’s behavior [1]\n\n### 1. Install the SDK\n\n```bash\npip install openai-agents\n```\n\nSet your API key:\n\n```bash\nexport OPENAI_API_KEY="your-api-key"\n```\n\n### 2. Define a tool\n\nTools let the agent take actions or retrieve information from external systems.\n\n```python\nfrom agents import function_tool\n\n@function_tool\ndef get_weather(city: str) -> str:\n    """Return the current weather for a city."""\n    # Replace this with a real weather API call.\n    return f"The weather in {city} is sunny and 22°C."\n```\n\n### 3. Create an agent\n\n```python\nfrom agents import Agent\n\nweather_agent = Agent(\n    name="Weather agent",\n    instructions=(\n        "You are a helpful weather assistant. "\n        "Use the get_weather tool w

In [20]:
print("="*70)
print("6) bob (finance_analyst) asks about the on-call runbook — ENGINEER-ONLY → DENIED")
print("="*70)
secure_agentic_rag("bob", "internal_runbook", "What should I do if the error rate spikes?")


6) bob (finance_analyst) asks about the on-call runbook — ENGINEER-ONLY → DENIED
👤 User: bob  (role: finance_analyst)
📄 Requested file: Internal Engineering On-Call Runbook
🚫 ACCESS DENIED — role 'finance_analyst' has no permission for this file.



"🚫 Access denied: your role ('finance_analyst') does not have permission to view 'Internal Engineering On-Call Runbook'."

**Notes on taking this further:**

- **Where roles come from.** This is an *allow-list* checked in application code. A real system would get the user's role from an identity provider (SSO/JWT claims) instead of a hardcoded `USERS` dict.
- **Access to part of a collection.** Here access is all-or-nothing per file, and the "10-K filings" file is the whole `10k_data` collection, so bob gets both Uber and Lyft. To allow only *some* chunks of a collection, put the permission check inside the vector search using **payload filters** (Qdrant supports this natively). Every point in `10k_data` carries `metadata.company` (`"Lyft, Inc."` / `"Uber Technologies, Inc."`) and most carry `metadata.fiscal_year`, so "Uber only" becomes a filter rather than an if-statement:

  ```python
  from qdrant_client import models

  hits = await rag_helpers._qdrant.query_points(
      collection_name="10k_data",
      query=embedding,
      limit=3,
      query_filter=models.Filter(must=[
          models.FieldCondition(key="metadata.company",
                                match=models.MatchValue(value="Uber Technologies, Inc.")),
      ]),
  )
  ```

  The difference matters: an if-statement after retrieval decides what to *show*, while a filter decides what retrieval is even allowed to *see*. Only the filter is safe once an LLM summarises the retrieved text, because anything retrieved can end up in the answer.
- **Audit trail.** Every access check is a natural place to log who asked for what and whether it was allowed, which is useful for compliance.

## 7. Inspect the Cache

List every entry in the semantic cache. Only the Section 5 (Path A) answers are here: time-sensitive answers, failed requests and all Section 6 (RBAC) answers are never stored.

In [21]:
print(f"Total cached entries: {len(cache.cache['questions'])}")
print(f"FAISS index size: {cache.index.ntotal}\n")

for i, (q, a) in enumerate(zip(cache.cache['questions'], cache.cache['response_text'])):
    print(f"[{i}] Q: {q}")
    print(f"    A: {a[:120]}...\n" if len(a) > 120 else f"    A: {a}\n")

Total cached entries: 4
FAISS index size: 4

[0] Q: What was Uber's revenue in 2021?
    A: Uber's revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]

[1] Q: How do I build an agent with the OpenAI Agents SDK?
    A: To build an agent with the OpenAI Agents SDK, define three core components:

1. **Model** — the language model responsib...

[2] Q: What are the most popular open-source LLMs?
    A: [1] Large language model
    They are the basis for many modern chatbots, such as ChatGPT, Claude, Gemini, Grok, and Dee...

[3] Q: Which open-source large language models are most widely used?
    A: [1] Best Open Source LLMs in 2026 (Updated for Q3) - QbitNeural
    For many AI teams, closed-source models like the GPT...



## Extend the System

Try one or more of these:

1. **Tune the threshold** — Try `threshold=0.2` (stricter) and `threshold=0.4` (looser). How do the hit rate and the answer quality change? Print the distance for a few question pairs, like notebook 002 does, before you pick a value.

2. **Cache TTL (time-to-live)** — Store a timestamp with each cache entry. Entries older than, say, 7 days should be ignored and fetched again.

3. **Split compound questions** — Before checking the cache, use an LLM call to split a question like *"What was Uber and Lyft revenue in 2021?"* into sub-questions. Check and fill the cache for each one separately.

4. **Cache analytics** — Track the hit rate, the average time for hits vs. misses, and the most common topics during a session.

5. **Combine RBAC with the cache** — Make Path B use a cache without leaking answers between roles. Hint: a hit should only count if the stored entry came from a file the current user is allowed to read (e.g. store the `file_id` with each entry, or keep one cache per file).